In [6]:
# ============================================================
# TGJU RETAIL IMAMI COIN SCRAPER
# JUPYTER WINDOWS SAFE VERSION
# ============================================================
#
# IMPORTANT:
# Playwright itself is NOT executed inside Jupyter's asyncio loop.
# Jupyter launches a completely separate Python worker process.
#
# Chrome: visible Google Chrome
# Date range:
#       1394/10/11
#       1405/04/05
#
# Output SQLite:
#       tgju_retail_sekee_history.db
#
# SQL table:
#       retail_sekee_history
# ============================================================

import sys
import subprocess
import os
from pathlib import Path
import textwrap


# ============================================================
# 1. INSTALL / UPDATE PLAYWRIGHT
# ============================================================

print("=" * 90)
print("CHECKING PLAYWRIGHT")
print("=" * 90)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "playwright"
])

print()
print("Playwright package is ready.")
print()


# ============================================================
# 2. CREATE STANDALONE WORKER SCRIPT
# ============================================================

WORKER_FILE = Path.cwd() / "tgju_scraper_worker.py"

worker_code = r'''
# ============================================================
# TGJU STANDALONE WORKER
# ============================================================

import os
import sys
import re
import time
import sqlite3
import traceback
from datetime import datetime

from playwright.sync_api import sync_playwright


# ============================================================
# CONFIG
# ============================================================

URL = "https://www.tgju.org/profile/sekeb/history"

START_DATE = "1394/06/11"
END_DATE   = "1405/06/05"

DATABASE_FILE = os.path.join(
    os.getcwd(),
    "tgju_retail_sekee_history.db"
)

TABLE_NAME = "retail_sekee_history"

HEADLESS = False

TIMEOUT_MS = 40000

PAGE_CHANGE_WAIT_MS = 1200

MAX_PAGES = 5000


# ============================================================
# DIGIT HELPERS
# ============================================================

PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
ARABIC_DIGITS  = "٠١٢٣٤٥٦٧٨٩"
ENGLISH_DIGITS = "0123456789"


def english_digits(value):

    if value is None:
        return None

    value = str(value)

    for fa, en in zip(
        PERSIAN_DIGITS,
        ENGLISH_DIGITS
    ):
        value = value.replace(fa, en)

    for ar, en in zip(
        ARABIC_DIGITS,
        ENGLISH_DIGITS
    ):
        value = value.replace(ar, en)

    return value.strip()


# ============================================================
# DATE NORMALIZATION
# ============================================================

def normalize_date(value):

    if not value:
        return None

    value = english_digits(value)

    nums = re.findall(
        r"\d+",
        value
    )

    if len(nums) < 3:
        return value.strip()

    y = int(nums[0])
    m = int(nums[1])
    d = int(nums[2])

    return f"{y:04d}/{m:02d}/{d:02d}"


def date_tuple(value):

    value = normalize_date(value)

    if not value:
        return None

    try:

        y, m, d = value.split("/")

        return (
            int(y),
            int(m),
            int(d)
        )

    except Exception:

        return None


START_TUPLE = date_tuple(
    START_DATE
)

END_TUPLE = date_tuple(
    END_DATE
)


def date_in_range(value):

    t = date_tuple(value)

    if t is None:
        return False

    return (
        START_TUPLE
        <= t
        <= END_TUPLE
    )


def date_before_start(value):

    t = date_tuple(value)

    if t is None:
        return False

    return t < START_TUPLE


# ============================================================
# NUMBER PARSING
# ============================================================

def parse_integer(value):

    if value is None:
        return None

    value = english_digits(value)

    value = (
        value
        .replace(",", "")
        .replace("٬", "")
        .replace("،", "")
        .replace(" ", "")
        .replace("\n", "")
        .strip()
    )

    if value in {
        "",
        "-",
        "–",
        "—",
        "null",
        "None"
    }:
        return None

    match = re.search(
        r"[-+]?\d+",
        value
    )

    if not match:
        return None

    try:
        return int(
            match.group()
        )

    except Exception:
        return None


def parse_percent(value):

    if value is None:
        return None

    value = english_digits(value)

    value = (
        value
        .replace("%", "")
        .replace("٪", "")
        .replace(",", "")
        .replace(" ", "")
        .strip()
    )

    if value in {
        "",
        "-",
        "–",
        "—"
    }:
        return None

    match = re.search(
        r"[-+]?\d+(?:\.\d+)?",
        value
    )

    if not match:
        return None

    try:

        return float(
            match.group()
        )

    except Exception:

        return None


# ============================================================
# DATABASE
# ============================================================

def open_database():

    conn = sqlite3.connect(
        DATABASE_FILE
    )

    conn.execute(
        "PRAGMA journal_mode=WAL"
    )

    conn.execute(
        "PRAGMA synchronous=NORMAL"
    )

    cursor = conn.cursor()

    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {TABLE_NAME}
        (
            id INTEGER PRIMARY KEY AUTOINCREMENT,

            open_price INTEGER,
            low_price INTEGER,
            high_price INTEGER,
            close_price INTEGER,

            change_amount INTEGER,
            change_percent REAL,

            gregorian_date TEXT NOT NULL,
            jalali_date TEXT NOT NULL,

            scraped_at TEXT NOT NULL,

            UNIQUE(
                jalali_date,
                gregorian_date
            )
        )
        """
    )

    conn.commit()

    return conn, cursor


# ============================================================
# SAVE ROW
# ============================================================

def save_record(
    cursor,
    record
):

    # INSERT OR REPLACE avoids schema problems
    # if the DB was already created by one of the
    # previous versions of this notebook.

    cursor.execute(
        f"""
        INSERT OR REPLACE INTO {TABLE_NAME}
        (
            id,

            open_price,
            low_price,
            high_price,
            close_price,

            change_amount,
            change_percent,

            gregorian_date,
            jalali_date,

            scraped_at
        )

        VALUES
        (
            (
                SELECT id
                FROM {TABLE_NAME}
                WHERE
                    jalali_date = ?
                    AND
                    gregorian_date = ?
                LIMIT 1
            ),

            ?, ?, ?, ?,
            ?, ?,
            ?, ?,
            ?
        )
        """,

        (
            record["jalali_date"],
            record["gregorian_date"],

            record["open_price"],
            record["low_price"],
            record["high_price"],
            record["close_price"],

            record["change_amount"],
            record["change_percent"],

            record["gregorian_date"],
            record["jalali_date"],

            datetime.now().isoformat(
                timespec="seconds"
            )
        )
    )


# ============================================================
# LOCATOR HELPERS
# ============================================================

def first_visible(
    page,
    selectors
):

    for selector in selectors:

        try:

            loc = page.locator(
                selector
            )

            count = loc.count()

            for i in range(count):

                candidate = loc.nth(i)

                try:

                    if candidate.is_visible():

                        return candidate

                except Exception:

                    pass

        except Exception:

            pass

    return None


# ============================================================
# FIND DATE INPUTS
# ============================================================

def find_from_input(page):

    return first_visible(
        page,
        [
            "#history-from",

            "input[name='history-from']",

            "input[placeholder='از تاریخ']",

            "input[placeholder*='از تاریخ']",

            "xpath=//input[contains(@placeholder,'از تاریخ')]",

            "xpath=//input[contains(@id,'history-from')]",

            "xpath=//input[contains(@id,'from')]"
        ]
    )


def find_to_input(page):

    return first_visible(
        page,
        [
            "#history-to",

            "input[name='history-to']",

            "input[placeholder='تا تاریخ']",

            "input[placeholder*='تا تاریخ']",

            "xpath=//input[contains(@placeholder,'تا تاریخ')]",

            "xpath=//input[contains(@id,'history-to')]",

            "xpath=//input[contains(@id,'to')]"
        ]
    )


# ============================================================
# SET DATE FIELD
# ============================================================

def set_date_field(
    page,
    locator,
    value,
    label
):

    if locator is None:

        print(
            f"WARNING: {label} field was not found."
        )

        return False


    print(
        f"Setting {label}: {value}"
    )


    try:

        locator.scroll_into_view_if_needed()

    except Exception:

        pass


    # --------------------------------------------------------
    # Remove readonly if necessary
    # --------------------------------------------------------

    try:

        locator.evaluate(
            """
            el => {
                el.removeAttribute('readonly');
                el.removeAttribute('disabled');
            }
            """
        )

    except Exception:

        pass


    # --------------------------------------------------------
    # Normal visible typing
    # --------------------------------------------------------

    try:

        locator.click(
            force=True
        )

        page.wait_for_timeout(
            300
        )

        locator.press(
            "Control+A"
        )

        locator.press(
            "Backspace"
        )

        locator.type(
            value,
            delay=70
        )

    except Exception as e:

        print(
            f"Normal typing fallback for {label}:",
            str(e)[:150]
        )


    # --------------------------------------------------------
    # Guarantee correct value using JS
    # --------------------------------------------------------

    try:

        locator.evaluate(
            """
            (el, newValue) => {

                const nativeInputValueSetter =
                    Object.getOwnPropertyDescriptor(
                        window.HTMLInputElement.prototype,
                        'value'
                    ).set;

                nativeInputValueSetter.call(
                    el,
                    newValue
                );

                el.dispatchEvent(
                    new Event(
                        'input',
                        {
                            bubbles: true
                        }
                    )
                );

                el.dispatchEvent(
                    new Event(
                        'change',
                        {
                            bubbles: true
                        }
                    )
                );
            }
            """,
            value
        )

    except Exception:

        pass


    # --------------------------------------------------------
    # Tab / Enter
    # --------------------------------------------------------

    try:

        locator.press(
            "Tab"
        )

    except Exception:

        pass


    page.wait_for_timeout(
        800
    )


    try:

        actual = english_digits(
            locator.input_value()
        )

        print(
            f"Actual {label} value: {actual}"
        )

    except Exception:

        pass


    return True


# ============================================================
# FIND HISTORY TABLE
# ============================================================

def find_history_table(page):

    # First try known selectors.

    for selector in [
        "#table-list",
        "table#table-list",
        "table.dataTable",
        "table.data-table"
    ]:

        try:

            loc = page.locator(
                selector
            )

            if loc.count() > 0:

                candidate = loc.first

                text = candidate.inner_text()

                if (
                    "بازگشایی" in text
                    or
                    "کمترین" in text
                    or
                    "تاریخ" in text
                ):

                    return candidate

        except Exception:

            pass


    # --------------------------------------------------------
    # Robust fallback:
    # inspect all visible tables and choose the one
    # containing TGJU's history headings.
    # --------------------------------------------------------

    tables = page.locator(
        "table"
    )

    for i in range(
        tables.count()
    ):

        try:

            table = tables.nth(i)

            if not table.is_visible():
                continue

            text = table.inner_text()

            if (
                "بازگشایی" in text
                and
                "کمترین" in text
                and
                "بیشترین" in text
                and
                "پایانی" in text
            ):

                return table

        except Exception:

            pass


    return None


# ============================================================
# SCRAPE CURRENT TABLE PAGE
# ============================================================

def scrape_current_page(page):

    table = find_history_table(
        page
    )

    if table is None:

        raise RuntimeError(
            "TGJU history table could not be found."
        )


    rows = table.locator(
        "tbody tr"
    )


    if rows.count() == 0:

        # Some HTML tables have no explicit tbody locator
        rows = table.locator(
            "tr"
        )


    results = []


    for i in range(
        rows.count()
    ):

        try:

            row = rows.nth(i)

            cells = row.locator(
                "td"
            )

            if cells.count() < 8:
                continue


            values = []


            for j in range(8):

                value = cells.nth(
                    j
                ).inner_text()

                value = english_digits(
                    value
                )

                values.append(
                    value
                )


            # =================================================
            # TGJU table:
            #
            # 0 = Open
            # 1 = Low
            # 2 = High
            # 3 = Close
            # 4 = Change amount
            # 5 = Change percent
            # 6 = Gregorian date
            # 7 = Jalali date
            # =================================================


            record = {

                "open_price":
                    parse_integer(
                        values[0]
                    ),

                "low_price":
                    parse_integer(
                        values[1]
                    ),

                "high_price":
                    parse_integer(
                        values[2]
                    ),

                "close_price":
                    parse_integer(
                        values[3]
                    ),

                "change_amount":
                    parse_integer(
                        values[4]
                    ),

                "change_percent":
                    parse_percent(
                        values[5]
                    ),

                "gregorian_date":
                    normalize_date(
                        values[6]
                    ),

                "jalali_date":
                    normalize_date(
                        values[7]
                    )
            }


            if (
                record["jalali_date"]
                and
                record["gregorian_date"]
            ):

                results.append(
                    record
                )


        except Exception as e:

            print(
                "Row parsing warning:",
                str(e)[:200]
            )


    return results


# ============================================================
# TABLE SIGNATURE
# ============================================================

def table_signature(page):

    try:

        records = scrape_current_page(
            page
        )

        if not records:
            return None

        first = records[0]
        last = records[-1]

        return (
            f"{first['jalali_date']}|"
            f"{first['close_price']}|"
            f"{last['jalali_date']}|"
            f"{last['close_price']}|"
            f"{len(records)}"
        )

    except Exception:

        return None


# ============================================================
# FIND NEXT BUTTON
# ============================================================

def find_next_button(page):

    selectors = [

        "#DataTables_Table_0_next",

        "button.dt-paging-button.next",

        "a.paginate_button.next",

        ".dataTables_paginate .next",

        ".dt-paging .next",

        ".pagination .next",

        "li.next a",

        "a[aria-label='Next']",

        "button[aria-label='Next']",

        "a[aria-label='next']",

        "button[aria-label='next']",

        "xpath=//a[contains(@class,'next')]",

        "xpath=//button[contains(@class,'next')]",

        "xpath=//*[self::a or self::button][contains(normalize-space(.),'بعدی')]"
    ]


    button = first_visible(
        page,
        selectors
    )

    if button is not None:

        return button


    # --------------------------------------------------------
    # Last fallback: inspect page controls
    # --------------------------------------------------------

    controls = page.locator(
        "a, button"
    )


    for i in range(
        controls.count()
    ):

        try:

            c = controls.nth(i)

            if not c.is_visible():
                continue

            cls = (
                c.get_attribute("class")
                or ""
            ).lower()

            aria = (
                c.get_attribute("aria-label")
                or ""
            ).lower()

            title = (
                c.get_attribute("title")
                or ""
            ).lower()

            txt = (
                c.inner_text()
                or ""
            ).strip().lower()


            if (
                "next" in cls
                or
                aria == "next"
                or
                title == "next"
                or
                txt == "بعدی"
            ):

                return c

        except Exception:

            pass


    return None


# ============================================================
# NEXT DISABLED?
# ============================================================

def next_disabled(button):

    if button is None:
        return True


    try:

        cls = (
            button.get_attribute("class")
            or ""
        ).lower()

        if "disabled" in cls:
            return True

    except Exception:

        pass


    try:

        aria = (
            button.get_attribute(
                "aria-disabled"
            )
            or ""
        ).lower()

        if aria == "true":
            return True

    except Exception:

        pass


    try:

        disabled = button.get_attribute(
            "disabled"
        )

        if disabled is not None:
            return True

    except Exception:

        pass


    try:

        parent_class = button.evaluate(
            """
            el => el.parentElement
                  ? String(el.parentElement.className)
                  : ''
            """
        ).lower()

        if "disabled" in parent_class:
            return True

    except Exception:

        pass


    return False


# ============================================================
# CLICK NEXT AND WAIT
# ============================================================

def next_page(
    page,
    old_signature
):

    button = find_next_button(
        page
    )


    if button is None:

        print(
            "NEXT button not found."
        )

        return False


    if next_disabled(
        button
    ):

        print(
            "NEXT button is disabled -> last page."
        )

        return False


    try:

        button.scroll_into_view_if_needed()

    except Exception:

        pass


    try:

        button.click(
            force=True
        )

    except Exception as e:

        print(
            "Normal NEXT click failed:",
            str(e)[:200]
        )

        try:

            button.evaluate(
                "el => el.click()"
            )

        except Exception:

            return False


    # --------------------------------------------------------
    # Wait for table content to change
    # --------------------------------------------------------

    for attempt in range(40):

        page.wait_for_timeout(
            350
        )

        new_signature = table_signature(
            page
        )

        if (
            new_signature
            and
            new_signature != old_signature
        ):

            page.wait_for_timeout(
                PAGE_CHANGE_WAIT_MS
            )

            return True


    print(
        "NEXT was clicked but table did not change."
    )

    return False


# ============================================================
# CLOSE POSSIBLE POPUPS
# ============================================================

def close_popups(page):

    candidates = [

        "button:has-text('خیر')",

        "button:has-text('نه')",

        "button:has-text('بستن')",

        ".modal .close",

        ".modal button.close",

        "button[aria-label='Close']"
    ]


    for selector in candidates:

        try:

            loc = page.locator(
                selector
            )

            for i in range(
                loc.count()
            ):

                candidate = loc.nth(i)

                try:

                    if candidate.is_visible():

                        candidate.click(
                            force=True,
                            timeout=1500
                        )

                        page.wait_for_timeout(
                            200
                        )

                except Exception:

                    pass

        except Exception:

            pass


# ============================================================
# DATABASE SUMMARY
# ============================================================

def print_database_summary(
    cursor
):

    cursor.execute(
        f"""
        SELECT COUNT(*)
        FROM {TABLE_NAME}
        """
    )

    count = cursor.fetchone()[0]


    cursor.execute(
        f"""
        SELECT
            MIN(jalali_date),
            MAX(jalali_date)
        FROM {TABLE_NAME}
        """
    )

    min_date, max_date = (
        cursor.fetchone()
    )


    print()
    print("=" * 90)
    print("DATABASE SUMMARY")
    print("=" * 90)

    print(
        "Rows:",
        count
    )

    print(
        "Minimum Jalali date:",
        min_date
    )

    print(
        "Maximum Jalali date:",
        max_date
    )

    print(
        "Database:",
        DATABASE_FILE
    )

    print(
        "Table:",
        TABLE_NAME
    )

    print("=" * 90)


# ============================================================
# MAIN SCRAPER
# ============================================================

def main():

    conn = None
    browser = None


    try:

        # ====================================================
        # DATABASE
        # ====================================================

        conn, cursor = open_database()


        print()
        print("=" * 90)
        print("DATABASE READY")
        print("=" * 90)

        print(
            "Database:",
            DATABASE_FILE
        )

        print(
            "Table:",
            TABLE_NAME
        )


        # ====================================================
        # PLAYWRIGHT
        # ====================================================

        print()
        print("=" * 90)
        print("STARTING PLAYWRIGHT")
        print("=" * 90)


        with sync_playwright() as p:

            print(
                "Opening installed Google Chrome..."
            )


            # =================================================
            # Uses the real installed Google Chrome.
            #
            # No ChromeDriver.
            # =================================================

            browser = p.chromium.launch(

                channel="chrome",

                headless=HEADLESS,

                args=[
                    "--start-maximized",
                    "--disable-notifications",
                    "--disable-popup-blocking"
                ]
            )


            context = browser.new_context(
                viewport=None,
                locale="fa-IR"
            )


            page = context.new_page()

            page.set_default_timeout(
                TIMEOUT_MS
            )


            # =================================================
            # OPEN TGJU
            # =================================================

            print()
            print("=" * 90)
            print("OPENING TGJU")
            print("=" * 90)

            print(
                URL
            )


            page.goto(
                URL,
                wait_until="domcontentloaded"
            )


            page.wait_for_timeout(
                5000
            )


            close_popups(
                page
            )


            # =================================================
            # TABLE CHECK
            # =================================================

            table = find_history_table(
                page
            )


            if table is None:

                raise RuntimeError(
                    "History table was not found on TGJU."
                )


            print()
            print(
                "TGJU history table found successfully."
            )


            # =================================================
            # DATE FIELDS
            # =================================================

            print()
            print("=" * 90)
            print("SETTING REQUESTED DATE RANGE")
            print("=" * 90)


            from_input = find_from_input(
                page
            )

            to_input = find_to_input(
                page
            )


            set_date_field(
                page,
                from_input,
                START_DATE,
                "FROM"
            )


            set_date_field(
                page,
                to_input,
                END_DATE,
                "TO"
            )


            # =================================================
            # Click outside datepicker
            # =================================================

            try:

                page.locator("body").click(
                    position={
                        "x": 20,
                        "y": 20
                    },
                    force=True
                )

            except Exception:

                pass


            # Trigger Enter on TO field too
            try:

                to_input.press(
                    "Enter"
                )

            except Exception:

                pass


            print()
            print(
                "Waiting for TGJU date filter..."
            )


            page.wait_for_timeout(
                5000
            )


            # =================================================
            # INITIAL TABLE INFO
            # =================================================

            initial_records = scrape_current_page(
                page
            )


            if initial_records:

                print()
                print(
                    "First visible Jalali date:",
                    initial_records[0][
                        "jalali_date"
                    ]
                )

                print(
                    "Last visible Jalali date:",
                    initial_records[-1][
                        "jalali_date"
                    ]
                )

                print(
                    "Rows on first page:",
                    len(initial_records)
                )


            # =================================================
            # PAGINATION
            # =================================================

            print()
            print("=" * 90)
            print("STARTING PAGINATION")
            print("=" * 90)


            page_number = 1

            total_seen = 0

            total_saved = 0

            seen_signatures = set()


            while (
                page_number
                <= MAX_PAGES
            ):

                current_signature = (
                    table_signature(
                        page
                    )
                )


                if not current_signature:

                    print(
                        "No valid table signature."
                    )

                    break


                if (
                    current_signature
                    in seen_signatures
                ):

                    print()
                    print(
                        "Repeated page detected."
                    )

                    print(
                        "Stopping to avoid an infinite loop."
                    )

                    break


                seen_signatures.add(
                    current_signature
                )


                records = scrape_current_page(
                    page
                )


                print()
                print(
                    "-" * 90
                )

                print(
                    f"PAGE {page_number}"
                )

                print(
                    "-" * 90
                )

                print(
                    "Rows detected:",
                    len(records)
                )


                if not records:

                    print(
                        "No rows found."
                    )

                    break


                page_saved = 0

                reached_older_data = False


                for record in records:

                    total_seen += 1


                    jdate = record[
                        "jalali_date"
                    ]


                    print(

                        f"{jdate}"
                        f" | {record['gregorian_date']}"
                        f" | O={record['open_price']}"
                        f" | L={record['low_price']}"
                        f" | H={record['high_price']}"
                        f" | C={record['close_price']}"
                    )


                    # =========================================
                    # EXTRA SAFETY:
                    #
                    # Even if TGJU's date filter does not work,
                    # ONLY requested dates are written to SQL.
                    # =========================================

                    if date_in_range(
                        jdate
                    ):

                        save_record(
                            cursor,
                            record
                        )

                        page_saved += 1

                        total_saved += 1


                    # =========================================
                    # Data is newest -> oldest.
                    #
                    # Once below 1394/10/11, stop.
                    # =========================================

                    if date_before_start(
                        jdate
                    ):

                        reached_older_data = True


                conn.commit()


                print()
                print(
                    "Saved this page:",
                    page_saved
                )

                print(
                    "Total saved this run:",
                    total_saved
                )


                # =============================================
                # Reached older than requested start date
                # =============================================

                if reached_older_data:

                    print()
                    print(
                        f"Reached date older than {START_DATE}."
                    )

                    print(
                        "Requested history is complete."
                    )

                    break


                # =============================================
                # NEXT PAGE
                # =============================================

                print()
                print(
                    "Clicking NEXT..."
                )


                moved = next_page(
                    page,
                    current_signature
                )


                if not moved:

                    print()
                    print(
                        "Reached final available page."
                    )

                    break


                page_number += 1


            # =================================================
            # FINAL SUMMARY
            # =================================================

            conn.commit()


            print()
            print("=" * 90)
            print("SCRAPING FINISHED")
            print("=" * 90)

            print(
                "Pages processed:",
                page_number
            )

            print(
                "Rows inspected:",
                total_seen
            )

            print(
                "Rows saved during this run:",
                total_saved
            )


            print_database_summary(
                cursor
            )


            # =================================================
            # SAMPLE
            # =================================================

            cursor.execute(
                f"""
                SELECT

                    jalali_date,
                    gregorian_date,

                    open_price,
                    low_price,
                    high_price,
                    close_price,

                    change_amount,
                    change_percent

                FROM {TABLE_NAME}

                ORDER BY
                    jalali_date DESC

                LIMIT 20
                """
            )


            samples = cursor.fetchall()


            print()
            print("=" * 90)
            print("SAMPLE SAVED ROWS")
            print("=" * 90)


            for row in samples:

                print(
                    row
                )


            print()
            print("=" * 90)
            print("DONE")
            print("=" * 90)


            # Keep final result visible for a moment
            page.wait_for_timeout(
                3000
            )


            browser.close()

            browser = None


    except Exception as e:

        print()
        print("=" * 90)
        print("ERROR")
        print("=" * 90)

        print(
            type(e).__name__
        )

        print(
            str(e)
        )

        print()
        print(
            traceback.format_exc()
        )


        if conn is not None:

            try:
                conn.commit()

            except Exception:
                pass


        # Leave browser briefly visible on error
        if browser is not None:

            try:
                time.sleep(5)

            except Exception:
                pass


        raise


    finally:

        if conn is not None:

            try:
                conn.close()

            except Exception:
                pass


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()
'''


WORKER_FILE.write_text(
    textwrap.dedent(worker_code),
    encoding="utf-8"
)


print("=" * 90)
print("WORKER CREATED")
print("=" * 90)

print(
    "Worker:",
    WORKER_FILE
)

print()


# ============================================================
# 3. RUN WORKER IN A COMPLETELY SEPARATE PYTHON PROCESS
# ============================================================

print("=" * 90)
print("STARTING TGJU SCRAPER IN SEPARATE PROCESS")
print("=" * 90)

print()
print("Chrome should open automatically.")
print()
print("Do NOT close the Chrome window while the scraper is working.")
print()


process = subprocess.Popen(

    [
        sys.executable,
        "-u",
        str(WORKER_FILE)
    ],

    cwd=str(Path.cwd()),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    encoding="utf-8",

    errors="replace",

    bufsize=1
)


# ============================================================
# 4. SHOW WORKER OUTPUT LIVE INSIDE JUPYTER
# ============================================================

for line in process.stdout:

    print(
        line,
        end=""
    )


return_code = process.wait()


print()
print("=" * 90)

if return_code == 0:

    print(
        "SCRAPER PROCESS FINISHED SUCCESSFULLY"
    )

else:

    print(
        f"SCRAPER PROCESS FAILED — EXIT CODE {return_code}"
    )

print("=" * 90)


# ============================================================
# 5. SHOW OUTPUT FILE LOCATION
# ============================================================

db_path = (
    Path.cwd()
    /
    "tgju_retail_sekeeb_history.db"
)


print()
print(
    "SQLite database:"
)

print(
    db_path
)

print()
print(
    "SQL table: retail_sekee_history"
)

CHECKING PLAYWRIGHT

Playwright package is ready.

WORKER CREATED
Worker: C:\Users\Lenovo\tgju_scraper_worker.py

STARTING TGJU SCRAPER IN SEPARATE PROCESS

Chrome should open automatically.

Do NOT close the Chrome window while the scraper is working.


DATABASE READY
Database: C:\Users\Lenovo\tgju_retail_sekee_history.db
Table: retail_sekee_history

STARTING PLAYWRIGHT
Opening installed Google Chrome...

OPENING TGJU
https://www.tgju.org/profile/sekeb/history

TGJU history table found successfully.

SETTING REQUESTED DATE RANGE
Setting FROM: 1394/06/11
Actual FROM value: 1394/06/11
Setting TO: 1405/06/05
Actual TO value: 1405/06/05

Waiting for TGJU date filter...

First visible Jalali date: 1405/06/05
Last visible Jalali date: 1405/04/30
Rows on first page: 30

STARTING PAGINATION

------------------------------------------------------------------------------------------
PAGE 1
------------------------------------------------------------------------------------------
Rows detected: 

# usd_dollar

In [7]:
# ============================================================
# TGJU RETAIL IMAMI COIN SCRAPER
# JUPYTER WINDOWS SAFE VERSION
# ============================================================
#
# IMPORTANT:
# Playwright itself is NOT executed inside Jupyter's asyncio loop.
# Jupyter launches a completely separate Python worker process.
#
# Chrome: visible Google Chrome
# Date range:
#       1394/10/11
#       1405/04/05
#
# Output SQLite:
#       tgju_retail_sekee_history.db
#
# SQL table:
#       retail_sekee_history
# ============================================================

import sys
import subprocess
import os
from pathlib import Path
import textwrap


# ============================================================
# 1. INSTALL / UPDATE PLAYWRIGHT
# ============================================================

print("=" * 90)
print("CHECKING PLAYWRIGHT")
print("=" * 90)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "playwright"
])

print()
print("Playwright package is ready.")
print()


# ============================================================
# 2. CREATE STANDALONE WORKER SCRIPT
# ============================================================

WORKER_FILE = Path.cwd() / "tgju_dollar_scraper_worker.py"

worker_code = r'''
# ============================================================
# TGJU STANDALONE WORKER
# ============================================================

import os
import sys
import re
import time
import sqlite3
import traceback
from datetime import datetime

from playwright.sync_api import sync_playwright


# ============================================================
# CONFIG
# ============================================================

URL = "https://www.tgju.org/profile/price_dollar_rl/history"

START_DATE = "1394/06/11"
END_DATE   = "1405/06/05"

DATABASE_FILE = os.path.join(
    os.getcwd(),
    "tgju_retail_dollar_history.db"
)

TABLE_NAME = "retail_sekee_history"

HEADLESS = False

TIMEOUT_MS = 40000

PAGE_CHANGE_WAIT_MS = 1200

MAX_PAGES = 5000


# ============================================================
# DIGIT HELPERS
# ============================================================

PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
ARABIC_DIGITS  = "٠١٢٣٤٥٦٧٨٩"
ENGLISH_DIGITS = "0123456789"


def english_digits(value):

    if value is None:
        return None

    value = str(value)

    for fa, en in zip(
        PERSIAN_DIGITS,
        ENGLISH_DIGITS
    ):
        value = value.replace(fa, en)

    for ar, en in zip(
        ARABIC_DIGITS,
        ENGLISH_DIGITS
    ):
        value = value.replace(ar, en)

    return value.strip()


# ============================================================
# DATE NORMALIZATION
# ============================================================

def normalize_date(value):

    if not value:
        return None

    value = english_digits(value)

    nums = re.findall(
        r"\d+",
        value
    )

    if len(nums) < 3:
        return value.strip()

    y = int(nums[0])
    m = int(nums[1])
    d = int(nums[2])

    return f"{y:04d}/{m:02d}/{d:02d}"


def date_tuple(value):

    value = normalize_date(value)

    if not value:
        return None

    try:

        y, m, d = value.split("/")

        return (
            int(y),
            int(m),
            int(d)
        )

    except Exception:

        return None


START_TUPLE = date_tuple(
    START_DATE
)

END_TUPLE = date_tuple(
    END_DATE
)


def date_in_range(value):

    t = date_tuple(value)

    if t is None:
        return False

    return (
        START_TUPLE
        <= t
        <= END_TUPLE
    )


def date_before_start(value):

    t = date_tuple(value)

    if t is None:
        return False

    return t < START_TUPLE


# ============================================================
# NUMBER PARSING
# ============================================================

def parse_integer(value):

    if value is None:
        return None

    value = english_digits(value)

    value = (
        value
        .replace(",", "")
        .replace("٬", "")
        .replace("،", "")
        .replace(" ", "")
        .replace("\n", "")
        .strip()
    )

    if value in {
        "",
        "-",
        "–",
        "—",
        "null",
        "None"
    }:
        return None

    match = re.search(
        r"[-+]?\d+",
        value
    )

    if not match:
        return None

    try:
        return int(
            match.group()
        )

    except Exception:
        return None


def parse_percent(value):

    if value is None:
        return None

    value = english_digits(value)

    value = (
        value
        .replace("%", "")
        .replace("٪", "")
        .replace(",", "")
        .replace(" ", "")
        .strip()
    )

    if value in {
        "",
        "-",
        "–",
        "—"
    }:
        return None

    match = re.search(
        r"[-+]?\d+(?:\.\d+)?",
        value
    )

    if not match:
        return None

    try:

        return float(
            match.group()
        )

    except Exception:

        return None


# ============================================================
# DATABASE
# ============================================================

def open_database():

    conn = sqlite3.connect(
        DATABASE_FILE
    )

    conn.execute(
        "PRAGMA journal_mode=WAL"
    )

    conn.execute(
        "PRAGMA synchronous=NORMAL"
    )

    cursor = conn.cursor()

    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {TABLE_NAME}
        (
            id INTEGER PRIMARY KEY AUTOINCREMENT,

            open_price INTEGER,
            low_price INTEGER,
            high_price INTEGER,
            close_price INTEGER,

            change_amount INTEGER,
            change_percent REAL,

            gregorian_date TEXT NOT NULL,
            jalali_date TEXT NOT NULL,

            scraped_at TEXT NOT NULL,

            UNIQUE(
                jalali_date,
                gregorian_date
            )
        )
        """
    )

    conn.commit()

    return conn, cursor


# ============================================================
# SAVE ROW
# ============================================================

def save_record(
    cursor,
    record
):

    # INSERT OR REPLACE avoids schema problems
    # if the DB was already created by one of the
    # previous versions of this notebook.

    cursor.execute(
        f"""
        INSERT OR REPLACE INTO {TABLE_NAME}
        (
            id,

            open_price,
            low_price,
            high_price,
            close_price,

            change_amount,
            change_percent,

            gregorian_date,
            jalali_date,

            scraped_at
        )

        VALUES
        (
            (
                SELECT id
                FROM {TABLE_NAME}
                WHERE
                    jalali_date = ?
                    AND
                    gregorian_date = ?
                LIMIT 1
            ),

            ?, ?, ?, ?,
            ?, ?,
            ?, ?,
            ?
        )
        """,

        (
            record["jalali_date"],
            record["gregorian_date"],

            record["open_price"],
            record["low_price"],
            record["high_price"],
            record["close_price"],

            record["change_amount"],
            record["change_percent"],

            record["gregorian_date"],
            record["jalali_date"],

            datetime.now().isoformat(
                timespec="seconds"
            )
        )
    )


# ============================================================
# LOCATOR HELPERS
# ============================================================

def first_visible(
    page,
    selectors
):

    for selector in selectors:

        try:

            loc = page.locator(
                selector
            )

            count = loc.count()

            for i in range(count):

                candidate = loc.nth(i)

                try:

                    if candidate.is_visible():

                        return candidate

                except Exception:

                    pass

        except Exception:

            pass

    return None


# ============================================================
# FIND DATE INPUTS
# ============================================================

def find_from_input(page):

    return first_visible(
        page,
        [
            "#history-from",

            "input[name='history-from']",

            "input[placeholder='از تاریخ']",

            "input[placeholder*='از تاریخ']",

            "xpath=//input[contains(@placeholder,'از تاریخ')]",

            "xpath=//input[contains(@id,'history-from')]",

            "xpath=//input[contains(@id,'from')]"
        ]
    )


def find_to_input(page):

    return first_visible(
        page,
        [
            "#history-to",

            "input[name='history-to']",

            "input[placeholder='تا تاریخ']",

            "input[placeholder*='تا تاریخ']",

            "xpath=//input[contains(@placeholder,'تا تاریخ')]",

            "xpath=//input[contains(@id,'history-to')]",

            "xpath=//input[contains(@id,'to')]"
        ]
    )


# ============================================================
# SET DATE FIELD
# ============================================================

def set_date_field(
    page,
    locator,
    value,
    label
):

    if locator is None:

        print(
            f"WARNING: {label} field was not found."
        )

        return False


    print(
        f"Setting {label}: {value}"
    )


    try:

        locator.scroll_into_view_if_needed()

    except Exception:

        pass


    # --------------------------------------------------------
    # Remove readonly if necessary
    # --------------------------------------------------------

    try:

        locator.evaluate(
            """
            el => {
                el.removeAttribute('readonly');
                el.removeAttribute('disabled');
            }
            """
        )

    except Exception:

        pass


    # --------------------------------------------------------
    # Normal visible typing
    # --------------------------------------------------------

    try:

        locator.click(
            force=True
        )

        page.wait_for_timeout(
            300
        )

        locator.press(
            "Control+A"
        )

        locator.press(
            "Backspace"
        )

        locator.type(
            value,
            delay=70
        )

    except Exception as e:

        print(
            f"Normal typing fallback for {label}:",
            str(e)[:150]
        )


    # --------------------------------------------------------
    # Guarantee correct value using JS
    # --------------------------------------------------------

    try:

        locator.evaluate(
            """
            (el, newValue) => {

                const nativeInputValueSetter =
                    Object.getOwnPropertyDescriptor(
                        window.HTMLInputElement.prototype,
                        'value'
                    ).set;

                nativeInputValueSetter.call(
                    el,
                    newValue
                );

                el.dispatchEvent(
                    new Event(
                        'input',
                        {
                            bubbles: true
                        }
                    )
                );

                el.dispatchEvent(
                    new Event(
                        'change',
                        {
                            bubbles: true
                        }
                    )
                );
            }
            """,
            value
        )

    except Exception:

        pass


    # --------------------------------------------------------
    # Tab / Enter
    # --------------------------------------------------------

    try:

        locator.press(
            "Tab"
        )

    except Exception:

        pass


    page.wait_for_timeout(
        800
    )


    try:

        actual = english_digits(
            locator.input_value()
        )

        print(
            f"Actual {label} value: {actual}"
        )

    except Exception:

        pass


    return True


# ============================================================
# FIND HISTORY TABLE
# ============================================================

def find_history_table(page):

    # First try known selectors.

    for selector in [
        "#table-list",
        "table#table-list",
        "table.dataTable",
        "table.data-table"
    ]:

        try:

            loc = page.locator(
                selector
            )

            if loc.count() > 0:

                candidate = loc.first

                text = candidate.inner_text()

                if (
                    "بازگشایی" in text
                    or
                    "کمترین" in text
                    or
                    "تاریخ" in text
                ):

                    return candidate

        except Exception:

            pass


    # --------------------------------------------------------
    # Robust fallback:
    # inspect all visible tables and choose the one
    # containing TGJU's history headings.
    # --------------------------------------------------------

    tables = page.locator(
        "table"
    )

    for i in range(
        tables.count()
    ):

        try:

            table = tables.nth(i)

            if not table.is_visible():
                continue

            text = table.inner_text()

            if (
                "بازگشایی" in text
                and
                "کمترین" in text
                and
                "بیشترین" in text
                and
                "پایانی" in text
            ):

                return table

        except Exception:

            pass


    return None


# ============================================================
# SCRAPE CURRENT TABLE PAGE
# ============================================================

def scrape_current_page(page):

    table = find_history_table(
        page
    )

    if table is None:

        raise RuntimeError(
            "TGJU history table could not be found."
        )


    rows = table.locator(
        "tbody tr"
    )


    if rows.count() == 0:

        # Some HTML tables have no explicit tbody locator
        rows = table.locator(
            "tr"
        )


    results = []


    for i in range(
        rows.count()
    ):

        try:

            row = rows.nth(i)

            cells = row.locator(
                "td"
            )

            if cells.count() < 8:
                continue


            values = []


            for j in range(8):

                value = cells.nth(
                    j
                ).inner_text()

                value = english_digits(
                    value
                )

                values.append(
                    value
                )


            # =================================================
            # TGJU table:
            #
            # 0 = Open
            # 1 = Low
            # 2 = High
            # 3 = Close
            # 4 = Change amount
            # 5 = Change percent
            # 6 = Gregorian date
            # 7 = Jalali date
            # =================================================


            record = {

                "open_price":
                    parse_integer(
                        values[0]
                    ),

                "low_price":
                    parse_integer(
                        values[1]
                    ),

                "high_price":
                    parse_integer(
                        values[2]
                    ),

                "close_price":
                    parse_integer(
                        values[3]
                    ),

                "change_amount":
                    parse_integer(
                        values[4]
                    ),

                "change_percent":
                    parse_percent(
                        values[5]
                    ),

                "gregorian_date":
                    normalize_date(
                        values[6]
                    ),

                "jalali_date":
                    normalize_date(
                        values[7]
                    )
            }


            if (
                record["jalali_date"]
                and
                record["gregorian_date"]
            ):

                results.append(
                    record
                )


        except Exception as e:

            print(
                "Row parsing warning:",
                str(e)[:200]
            )


    return results


# ============================================================
# TABLE SIGNATURE
# ============================================================

def table_signature(page):

    try:

        records = scrape_current_page(
            page
        )

        if not records:
            return None

        first = records[0]
        last = records[-1]

        return (
            f"{first['jalali_date']}|"
            f"{first['close_price']}|"
            f"{last['jalali_date']}|"
            f"{last['close_price']}|"
            f"{len(records)}"
        )

    except Exception:

        return None


# ============================================================
# FIND NEXT BUTTON
# ============================================================

def find_next_button(page):

    selectors = [

        "#DataTables_Table_0_next",

        "button.dt-paging-button.next",

        "a.paginate_button.next",

        ".dataTables_paginate .next",

        ".dt-paging .next",

        ".pagination .next",

        "li.next a",

        "a[aria-label='Next']",

        "button[aria-label='Next']",

        "a[aria-label='next']",

        "button[aria-label='next']",

        "xpath=//a[contains(@class,'next')]",

        "xpath=//button[contains(@class,'next')]",

        "xpath=//*[self::a or self::button][contains(normalize-space(.),'بعدی')]"
    ]


    button = first_visible(
        page,
        selectors
    )

    if button is not None:

        return button


    # --------------------------------------------------------
    # Last fallback: inspect page controls
    # --------------------------------------------------------

    controls = page.locator(
        "a, button"
    )


    for i in range(
        controls.count()
    ):

        try:

            c = controls.nth(i)

            if not c.is_visible():
                continue

            cls = (
                c.get_attribute("class")
                or ""
            ).lower()

            aria = (
                c.get_attribute("aria-label")
                or ""
            ).lower()

            title = (
                c.get_attribute("title")
                or ""
            ).lower()

            txt = (
                c.inner_text()
                or ""
            ).strip().lower()


            if (
                "next" in cls
                or
                aria == "next"
                or
                title == "next"
                or
                txt == "بعدی"
            ):

                return c

        except Exception:

            pass


    return None


# ============================================================
# NEXT DISABLED?
# ============================================================

def next_disabled(button):

    if button is None:
        return True


    try:

        cls = (
            button.get_attribute("class")
            or ""
        ).lower()

        if "disabled" in cls:
            return True

    except Exception:

        pass


    try:

        aria = (
            button.get_attribute(
                "aria-disabled"
            )
            or ""
        ).lower()

        if aria == "true":
            return True

    except Exception:

        pass


    try:

        disabled = button.get_attribute(
            "disabled"
        )

        if disabled is not None:
            return True

    except Exception:

        pass


    try:

        parent_class = button.evaluate(
            """
            el => el.parentElement
                  ? String(el.parentElement.className)
                  : ''
            """
        ).lower()

        if "disabled" in parent_class:
            return True

    except Exception:

        pass


    return False


# ============================================================
# CLICK NEXT AND WAIT
# ============================================================

def next_page(
    page,
    old_signature
):

    button = find_next_button(
        page
    )


    if button is None:

        print(
            "NEXT button not found."
        )

        return False


    if next_disabled(
        button
    ):

        print(
            "NEXT button is disabled -> last page."
        )

        return False


    try:

        button.scroll_into_view_if_needed()

    except Exception:

        pass


    try:

        button.click(
            force=True
        )

    except Exception as e:

        print(
            "Normal NEXT click failed:",
            str(e)[:200]
        )

        try:

            button.evaluate(
                "el => el.click()"
            )

        except Exception:

            return False


    # --------------------------------------------------------
    # Wait for table content to change
    # --------------------------------------------------------

    for attempt in range(40):

        page.wait_for_timeout(
            350
        )

        new_signature = table_signature(
            page
        )

        if (
            new_signature
            and
            new_signature != old_signature
        ):

            page.wait_for_timeout(
                PAGE_CHANGE_WAIT_MS
            )

            return True


    print(
        "NEXT was clicked but table did not change."
    )

    return False


# ============================================================
# CLOSE POSSIBLE POPUPS
# ============================================================

def close_popups(page):

    candidates = [

        "button:has-text('خیر')",

        "button:has-text('نه')",

        "button:has-text('بستن')",

        ".modal .close",

        ".modal button.close",

        "button[aria-label='Close']"
    ]


    for selector in candidates:

        try:

            loc = page.locator(
                selector
            )

            for i in range(
                loc.count()
            ):

                candidate = loc.nth(i)

                try:

                    if candidate.is_visible():

                        candidate.click(
                            force=True,
                            timeout=1500
                        )

                        page.wait_for_timeout(
                            200
                        )

                except Exception:

                    pass

        except Exception:

            pass


# ============================================================
# DATABASE SUMMARY
# ============================================================

def print_database_summary(
    cursor
):

    cursor.execute(
        f"""
        SELECT COUNT(*)
        FROM {TABLE_NAME}
        """
    )

    count = cursor.fetchone()[0]


    cursor.execute(
        f"""
        SELECT
            MIN(jalali_date),
            MAX(jalali_date)
        FROM {TABLE_NAME}
        """
    )

    min_date, max_date = (
        cursor.fetchone()
    )


    print()
    print("=" * 90)
    print("DATABASE SUMMARY")
    print("=" * 90)

    print(
        "Rows:",
        count
    )

    print(
        "Minimum Jalali date:",
        min_date
    )

    print(
        "Maximum Jalali date:",
        max_date
    )

    print(
        "Database:",
        DATABASE_FILE
    )

    print(
        "Table:",
        TABLE_NAME
    )

    print("=" * 90)


# ============================================================
# MAIN SCRAPER
# ============================================================

def main():

    conn = None
    browser = None


    try:

        # ====================================================
        # DATABASE
        # ====================================================

        conn, cursor = open_database()


        print()
        print("=" * 90)
        print("DATABASE READY")
        print("=" * 90)

        print(
            "Database:",
            DATABASE_FILE
        )

        print(
            "Table:",
            TABLE_NAME
        )


        # ====================================================
        # PLAYWRIGHT
        # ====================================================

        print()
        print("=" * 90)
        print("STARTING PLAYWRIGHT")
        print("=" * 90)


        with sync_playwright() as p:

            print(
                "Opening installed Google Chrome..."
            )


            # =================================================
            # Uses the real installed Google Chrome.
            #
            # No ChromeDriver.
            # =================================================

            browser = p.chromium.launch(

                channel="chrome",

                headless=HEADLESS,

                args=[
                    "--start-maximized",
                    "--disable-notifications",
                    "--disable-popup-blocking"
                ]
            )


            context = browser.new_context(
                viewport=None,
                locale="fa-IR"
            )


            page = context.new_page()

            page.set_default_timeout(
                TIMEOUT_MS
            )


            # =================================================
            # OPEN TGJU
            # =================================================

            print()
            print("=" * 90)
            print("OPENING TGJU")
            print("=" * 90)

            print(
                URL
            )


            page.goto(
                URL,
                wait_until="domcontentloaded"
            )


            page.wait_for_timeout(
                5000
            )


            close_popups(
                page
            )


            # =================================================
            # TABLE CHECK
            # =================================================

            table = find_history_table(
                page
            )


            if table is None:

                raise RuntimeError(
                    "History table was not found on TGJU."
                )


            print()
            print(
                "TGJU history table found successfully."
            )


            # =================================================
            # DATE FIELDS
            # =================================================

            print()
            print("=" * 90)
            print("SETTING REQUESTED DATE RANGE")
            print("=" * 90)


            from_input = find_from_input(
                page
            )

            to_input = find_to_input(
                page
            )


            set_date_field(
                page,
                from_input,
                START_DATE,
                "FROM"
            )


            set_date_field(
                page,
                to_input,
                END_DATE,
                "TO"
            )


            # =================================================
            # Click outside datepicker
            # =================================================

            try:

                page.locator("body").click(
                    position={
                        "x": 20,
                        "y": 20
                    },
                    force=True
                )

            except Exception:

                pass


            # Trigger Enter on TO field too
            try:

                to_input.press(
                    "Enter"
                )

            except Exception:

                pass


            print()
            print(
                "Waiting for TGJU date filter..."
            )


            page.wait_for_timeout(
                5000
            )


            # =================================================
            # INITIAL TABLE INFO
            # =================================================

            initial_records = scrape_current_page(
                page
            )


            if initial_records:

                print()
                print(
                    "First visible Jalali date:",
                    initial_records[0][
                        "jalali_date"
                    ]
                )

                print(
                    "Last visible Jalali date:",
                    initial_records[-1][
                        "jalali_date"
                    ]
                )

                print(
                    "Rows on first page:",
                    len(initial_records)
                )


            # =================================================
            # PAGINATION
            # =================================================

            print()
            print("=" * 90)
            print("STARTING PAGINATION")
            print("=" * 90)


            page_number = 1

            total_seen = 0

            total_saved = 0

            seen_signatures = set()


            while (
                page_number
                <= MAX_PAGES
            ):

                current_signature = (
                    table_signature(
                        page
                    )
                )


                if not current_signature:

                    print(
                        "No valid table signature."
                    )

                    break


                if (
                    current_signature
                    in seen_signatures
                ):

                    print()
                    print(
                        "Repeated page detected."
                    )

                    print(
                        "Stopping to avoid an infinite loop."
                    )

                    break


                seen_signatures.add(
                    current_signature
                )


                records = scrape_current_page(
                    page
                )


                print()
                print(
                    "-" * 90
                )

                print(
                    f"PAGE {page_number}"
                )

                print(
                    "-" * 90
                )

                print(
                    "Rows detected:",
                    len(records)
                )


                if not records:

                    print(
                        "No rows found."
                    )

                    break


                page_saved = 0

                reached_older_data = False


                for record in records:

                    total_seen += 1


                    jdate = record[
                        "jalali_date"
                    ]


                    print(

                        f"{jdate}"
                        f" | {record['gregorian_date']}"
                        f" | O={record['open_price']}"
                        f" | L={record['low_price']}"
                        f" | H={record['high_price']}"
                        f" | C={record['close_price']}"
                    )


                    # =========================================
                    # EXTRA SAFETY:
                    #
                    # Even if TGJU's date filter does not work,
                    # ONLY requested dates are written to SQL.
                    # =========================================

                    if date_in_range(
                        jdate
                    ):

                        save_record(
                            cursor,
                            record
                        )

                        page_saved += 1

                        total_saved += 1


                    # =========================================
                    # Data is newest -> oldest.
                    #
                    # Once below 1394/10/11, stop.
                    # =========================================

                    if date_before_start(
                        jdate
                    ):

                        reached_older_data = True


                conn.commit()


                print()
                print(
                    "Saved this page:",
                    page_saved
                )

                print(
                    "Total saved this run:",
                    total_saved
                )


                # =============================================
                # Reached older than requested start date
                # =============================================

                if reached_older_data:

                    print()
                    print(
                        f"Reached date older than {START_DATE}."
                    )

                    print(
                        "Requested history is complete."
                    )

                    break


                # =============================================
                # NEXT PAGE
                # =============================================

                print()
                print(
                    "Clicking NEXT..."
                )


                moved = next_page(
                    page,
                    current_signature
                )


                if not moved:

                    print()
                    print(
                        "Reached final available page."
                    )

                    break


                page_number += 1


            # =================================================
            # FINAL SUMMARY
            # =================================================

            conn.commit()


            print()
            print("=" * 90)
            print("SCRAPING FINISHED")
            print("=" * 90)

            print(
                "Pages processed:",
                page_number
            )

            print(
                "Rows inspected:",
                total_seen
            )

            print(
                "Rows saved during this run:",
                total_saved
            )


            print_database_summary(
                cursor
            )


            # =================================================
            # SAMPLE
            # =================================================

            cursor.execute(
                f"""
                SELECT

                    jalali_date,
                    gregorian_date,

                    open_price,
                    low_price,
                    high_price,
                    close_price,

                    change_amount,
                    change_percent

                FROM {TABLE_NAME}

                ORDER BY
                    jalali_date DESC

                LIMIT 20
                """
            )


            samples = cursor.fetchall()


            print()
            print("=" * 90)
            print("SAMPLE SAVED ROWS")
            print("=" * 90)


            for row in samples:

                print(
                    row
                )


            print()
            print("=" * 90)
            print("DONE")
            print("=" * 90)


            # Keep final result visible for a moment
            page.wait_for_timeout(
                3000
            )


            browser.close()

            browser = None


    except Exception as e:

        print()
        print("=" * 90)
        print("ERROR")
        print("=" * 90)

        print(
            type(e).__name__
        )

        print(
            str(e)
        )

        print()
        print(
            traceback.format_exc()
        )


        if conn is not None:

            try:
                conn.commit()

            except Exception:
                pass


        # Leave browser briefly visible on error
        if browser is not None:

            try:
                time.sleep(5)

            except Exception:
                pass


        raise


    finally:

        if conn is not None:

            try:
                conn.close()

            except Exception:
                pass


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()
'''


WORKER_FILE.write_text(
    textwrap.dedent(worker_code),
    encoding="utf-8"
)


print("=" * 90)
print("WORKER CREATED")
print("=" * 90)

print(
    "Worker:",
    WORKER_FILE
)

print()


# ============================================================
# 3. RUN WORKER IN A COMPLETELY SEPARATE PYTHON PROCESS
# ============================================================

print("=" * 90)
print("STARTING TGJU SCRAPER IN SEPARATE PROCESS")
print("=" * 90)

print()
print("Chrome should open automatically.")
print()
print("Do NOT close the Chrome window while the scraper is working.")
print()


process = subprocess.Popen(

    [
        sys.executable,
        "-u",
        str(WORKER_FILE)
    ],

    cwd=str(Path.cwd()),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    encoding="utf-8",

    errors="replace",

    bufsize=1
)


# ============================================================
# 4. SHOW WORKER OUTPUT LIVE INSIDE JUPYTER
# ============================================================

for line in process.stdout:

    print(
        line,
        end=""
    )


return_code = process.wait()


print()
print("=" * 90)

if return_code == 0:

    print(
        "SCRAPER PROCESS FINISHED SUCCESSFULLY"
    )

else:

    print(
        f"SCRAPER PROCESS FAILED — EXIT CODE {return_code}"
    )

print("=" * 90)


# ============================================================
# 5. SHOW OUTPUT FILE LOCATION
# ============================================================

db_path = (
    Path.cwd()
    /
    "tgju_retail_dollar_history.db"
)


print()
print(
    "SQLite database:"
)

print(
    db_path
)

print()
print(
    "SQL table: retail_sekee_history"
)

CHECKING PLAYWRIGHT

Playwright package is ready.

WORKER CREATED
Worker: C:\Users\Lenovo\tgju_dollar_scraper_worker.py

STARTING TGJU SCRAPER IN SEPARATE PROCESS

Chrome should open automatically.

Do NOT close the Chrome window while the scraper is working.


DATABASE READY
Database: C:\Users\Lenovo\tgju_retail_dollar_history.db
Table: retail_sekee_history

STARTING PLAYWRIGHT
Opening installed Google Chrome...

OPENING TGJU
https://www.tgju.org/profile/price_dollar_rl/history

TGJU history table found successfully.

SETTING REQUESTED DATE RANGE
Setting FROM: 1394/06/11
Actual FROM value: 1394/06/11
Setting TO: 1405/06/05
Actual TO value: 1405/06/05

Waiting for TGJU date filter...

First visible Jalali date: 1405/06/05
Last visible Jalali date: 1405/04/30
Rows on first page: 30

STARTING PAGINATION

------------------------------------------------------------------------------------------
PAGE 1
----------------------------------------------------------------------------------------